# Module 9 — Feature Selection: Mutual Information, LASSO & SHAP

> **Track 1 · Data Science Foundations**  
> **Difficulty**: Intermediate → Advanced  
> **Runtime**: ~5 minutes  
> **Key Libraries**: Scikit-learn, SHAP, XGBoost, NumPy, Pandas, Plotly, Seaborn  
> **Prerequisite**: Module 8 (Hyperparameter Tuning)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic High-Dimensional Dataset](#3-synthetic-high-dimensional-dataset)
4. [Filter Methods: Mutual Information](#4-filter-methods-mutual-information)
5. [Wrapper Methods: Recursive Feature Elimination (RFE)](#5-wrapper-methods-rfe)
6. [Embedded Methods: LASSO Regularization](#6-embedded-methods-lasso)
7. [Model-Based: Feature Importance (XGBoost)](#7-model-based-feature-importance)
8. [SHAP Values: Game-Theoretic Feature Attribution](#8-shap-values)
9. [Comparison & Selection](#9-comparison--selection)
10. [Key Business Takeaway](#10-key-business-takeaway)

---
## §1 · Business Context

### Why does feature selection matter at Revolut?

The fraud-detection pipeline at Revolut ingests **500+ features** per transaction:
- Transaction metadata (amount, time, merchant)
- User behavior (velocity, geo, device)
- Historical aggregates (7-day/30-day rolling stats)

**The problem**: most of these features are **redundant or noisy**, leading to:

| Issue | Impact |
|-------|--------|
| Slow inference | 200 ms latency → missed fraud window |
| Overfitting | Model memorizes noise, poor generalization |
| High storage | 500 features × 10M txns/day = 20 GB/day |
| Poor interpretability | Compliance can't explain why a txn was flagged |

**The solution**: reduce 500 → 50 features while preserving 98% of model performance.

In this notebook we will:
1. Generate a 50-feature synthetic dataset with known important features.
2. Apply four feature-selection methods: mutual information, RFE, LASSO, SHAP.
3. Compare which features each method selects.
4. Measure model performance before and after selection.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import time
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LassoCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import (
    mutual_info_classif, RFE, SelectFromModel
)
from sklearn.metrics import roc_auc_score, accuracy_score
from xgboost import XGBClassifier

# ── SHAP ─────────────────────────────────────────────────────────
import shap

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")
print(f"  SHAP {shap.__version__}")

---
## §3 · Synthetic High-Dimensional Dataset

We simulate **50,000 samples** with **50 features**, but only **10 are truly important**.

In [ ]:
# ── Configuration ────────────────────────────────────────────────
N_SAMPLES = 50_000
N_FEATURES = 50
N_IMPORTANT = 10

print(f"Generating high-dimensional dataset:")
print(f"  Samples    : {N_SAMPLES:,}")
print(f"  Features   : {N_FEATURES}")
print(f"  Important  : {N_IMPORTANT} (20%)")

# ── Generate features ────────────────────────────────────────────
X = np.random.randn(N_SAMPLES, N_FEATURES)

# ── Create target with known important features ──────────────────
# Features 0–9 are important; 10–49 are noise
important_features = X[:, :N_IMPORTANT]
weights = np.array([2.0, -1.5, 1.8, 0.5, -0.8, 1.2, -1.0, 0.7, -0.5, 1.0])

log_odds = important_features @ weights + np.random.normal(0, 1, N_SAMPLES)
prob = 1 / (1 + np.exp(-log_odds))
y = (prob > 0.5).astype(int)

# Create feature names
feature_names = [f"important_{i}" if i < N_IMPORTANT else f"noise_{i}" for i in range(N_FEATURES)]
df = pd.DataFrame(X, columns=feature_names)
df["target"] = y

print(f"\n✓ Shape: {df.shape}")
print(f"  Positive rate: {y.mean() * 100:.1f}%")
print(f"  Important features: {feature_names[:N_IMPORTANT]}")

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
print(f"\nTrain: {len(y_train):,}  |  Test: {len(y_test):,}")

---
## §4 · Filter Methods: Mutual Information

**Mutual Information** measures the dependence between a feature and the target.
- MI = 0: feature and target are independent.
- MI > 0: feature provides information about the target.

In [ ]:
# Compute mutual information for all features
mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
mi_df = pd.DataFrame({
    "Feature": feature_names,
    "MI Score": mi_scores,
}).sort_values("MI Score", ascending=False)

mi_df["Is Important"] = mi_df["Feature"].str.startswith("important")

print("=" * 70)
print("MUTUAL INFORMATION FEATURE RANKING")
print("=" * 70)
print(mi_df.head(20).to_string(index=False))

# Select top 20 features
top_n = 20
selected_mi = mi_df.head(top_n)["Feature"].tolist()
print(f"\n✓ Selected top {top_n} features by MI")
print(f"  True important features in selection: {sum(1 for f in selected_mi if f.startswith('important'))}/{N_IMPORTANT}")

# Visualize
fig = px.bar(
    mi_df.head(25),
    x="MI Score",
    y="Feature",
    orientation="h",
    color="Is Important",
    color_discrete_map={True: "#00CC96", False: "#EF553B"},
    title="Top 25 Features by Mutual Information (green = truly important)",
)
fig.update_layout(height=550, yaxis=dict(autorange="reversed"))
fig.show()

---
## §5 · Wrapper Methods: Recursive Feature Elimination (RFE)

RFE trains a model, removes the least important feature, and repeats.

In [ ]:
# RFE with Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rfe = RFE(estimator=rf, n_features_to_select=20, step=5, verbose=0)

print("Running RFE with Random Forest …")
t0 = time.time()
rfe.fit(X_train, y_train)
time_rfe = time.time() - t0

print(f"✓ Completed in {time_rfe:.1f}s")

# Get selected features
selected_rfe = [feature_names[i] for i in range(N_FEATURES) if rfe.support_[i]]
print(f"\nSelected {len(selected_rfe)} features:")
for i, f in enumerate(selected_rfe[:15], 1):
    marker = "✓" if f.startswith("important") else "✗"
    print(f"  {i:2d}. {f:<20s} {marker}")

print(f"\nTrue important features in selection: {sum(1 for f in selected_rfe if f.startswith('important'))}/{N_IMPORTANT}")

# RFE ranking
rfe_ranking = pd.DataFrame({
    "Feature": feature_names,
    "RFE Rank": rfe.ranking_,
}).sort_values("RFE Rank")

print("\nRFE Ranking (1 = selected, higher = eliminated earlier):")
print(rfe_ranking.head(15).to_string(index=False))

---
## §6 · Embedded Methods: LASSO Regularization

LASSO (L1 regularization) **shrinks coefficients to exactly zero**, performing automatic feature selection.

In [ ]:
# Standardize features (important for LASSO)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# LASSO with cross-validated alpha
lasso = LassoCV(cv=5, random_state=42, max_iter=5000)
lasso.fit(X_train_scaled, y_train)

print(f"LASSO Results:")
print(f"  Optimal α    : {lasso.alpha_:.4f}")
print(f"  Non-zero coefficients: {(lasso.coef_ != 0).sum()}/{N_FEATURES}")

# Features selected by LASSO
lasso_features = []
lasso_coefs = []
for name, coef in zip(feature_names, lasso.coef_):
    if abs(coef) > 1e-6:
        lasso_features.append(name)
        lasso_coefs.append(coef)

lasso_df = pd.DataFrame({
    "Feature": lasso_features,
    "Coefficient": lasso_coefs,
    "Abs Coef": np.abs(lasso_coefs),
}).sort_values("Abs Coef", ascending=False)

print(f"\nFeatures selected by LASSO ({len(lasso_df)}):")
print(lasso_df.to_string(index=False))

print(f"\nTrue important features in selection: {sum(1 for f in lasso_features if f.startswith('important'))}/{N_IMPORTANT}")

# Visualize
fig = px.bar(
    lasso_df,
    x="Coefficient",
    y="Feature",
    orientation="h",
    color="Coefficient",
    color_continuous_scale="RdBu",
    title="LASSO Coefficients (non-zero = selected)",
)
fig.update_layout(height=450, yaxis=dict(autorange="reversed"))
fig.show()

---
## §7 · Model-Based: Feature Importance (XGBoost)

Tree-based models provide **built-in feature importance** based on how often features are used for splits.

In [ ]:
# Train XGBoost
xgb = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                     random_state=42, eval_metric="logloss", use_label_encoder=False)
xgb.fit(X_train, y_train)

# Get feature importance
importance_scores = xgb.feature_importances_
xgb_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance_scores,
}).sort_values("Importance", ascending=False)

xgb_df["Is Important"] = xgb_df["Feature"].str.startswith("important")

print("=" * 70)
print("XGBOOST FEATURE IMPORTANCE")
print("=" * 70)
print(xgb_df.head(20).to_string(index=False))

# Select top 20
selected_xgb = xgb_df.head(20)["Feature"].tolist()
print(f"\nTrue important features in top 20: {sum(1 for f in selected_xgb if f.startswith('important'))}/{N_IMPORTANT}")

# Visualize
fig = px.bar(
    xgb_df.head(25),
    x="Importance",
    y="Feature",
    orientation="h",
    color="Is Important",
    color_discrete_map={True: "#00CC96", False: "#EF553B"},
    title="Top 25 Features by XGBoost Importance (green = truly important)",
)
fig.update_layout(height=550, yaxis=dict(autorange="reversed"))
fig.show()

---
## §8 · SHAP Values: Game-Theoretic Feature Attribution

**SHAP (SHapley Additive exPlanations)** assigns each feature an importance value
for a **specific prediction**, based on cooperative game theory.

In [ ]:
# Compute SHAP values for XGBoost model
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)

print(f"SHAP values computed for {len(X_test):,} test samples")
print(f"  Shape: {shap_values.shape} (samples × features)")

# Global feature importance (mean absolute SHAP)
shap_importance = pd.DataFrame({
    "Feature": feature_names,
    "Mean |SHAP|": np.abs(shap_values).mean(axis=0),
}).sort_values("Mean |SHAP|", ascending=False)

shap_importance["Is Important"] = shap_importance["Feature"].str.startswith("important")

print("\n" + "=" * 70)
print("SHAP GLOBAL FEATURE IMPORTANCE")
print("=" * 70)
print(shap_importance.head(20).to_string(index=False))

selected_shap = shap_importance.head(20)["Feature"].tolist()
print(f"\nTrue important features in top 20: {sum(1 for f in selected_shap if f.startswith('important'))}/{N_IMPORTANT}")

In [ ]:
# SHAP summary plot (built-in)
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test, feature_names=feature_names, max_display=20, show=False)
plt.title("SHAP Summary Plot (Top 20 Features)")
plt.tight_layout()
plt.show()

print("💡 SHAP shows both feature importance AND direction of effect:")
print("   - Red (high feature value) pushing right = positive contribution")
print("   - Blue (low feature value) pushing left = negative contribution")

In [ ]:
# SHAP bar plot (global importance)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, feature_names=feature_names, max_display=20, plot_type="bar", show=False)
plt.title("SHAP Feature Importance (Mean |SHAP|)")
plt.tight_layout()
plt.show()

---
## §9 · Comparison & Selection

### 9.1 — Venn Diagram: Which Features Were Selected?

In [ ]:
# Compile selected features from each method
selections = {
    "Mutual Information": set(selected_mi),
    "RFE (Random Forest)": set(selected_rfe),
    "LASSO": set(lasso_features[:20]) if len(lasso_features) >= 20 else set(lasso_features),
    "XGBoost Importance": set(selected_xgb),
    "SHAP": set(selected_shap),
}

# Count how many methods selected each feature
feature_votes = pd.DataFrame({
    "Feature": feature_names,
    "Votes": [sum(f in s for s in selections.values()) for f in feature_names],
    "Is Important": [f.startswith("important") for f in feature_names],
}).sort_values("Votes", ascending=False)

print("=" * 70)
print("FEATURE SELECTION CONSENSUS")
print("=" * 70)
print(feature_votes.head(20).to_string(index=False))

print(f"\nFeatures selected by ALL 5 methods:")
unanimous = feature_votes[feature_votes["Votes"] == 5]["Feature"].tolist()
print(f"  {unanimous}")

print(f"\nFeatures selected by ≥4 methods:")
strong_consensus = feature_votes[feature_votes["Votes"] >= 4]["Feature"].tolist()
print(f"  {strong_consensus}")

In [ ]:
# ── 9.2 Performance Comparison ───────────────────────────────────
# Train models with different feature sets and compare AUC
def evaluate_features(selected_features, method_name):
    # Get indices of selected features
    selected_idx = [feature_names.index(f) for f in selected_features if f in feature_names]
    X_train_sel = X_train[:, selected_idx]
    X_test_sel = X_test[:, selected_idx]
    
    # Train XGBoost
    model = XGBClassifier(n_estimators=100, max_depth=5, random_state=42,
                           eval_metric="logloss", use_label_encoder=False)
    model.fit(X_train_sel, y_train)
    y_prob = model.predict_proba(X_test_sel)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    return auc, len(selected_idx)

results = []

# All features
auc_all, n_all = evaluate_features(feature_names, "All Features")
results.append({"Method": "All Features", "N Features": n_all, "Test AUC": auc_all})

# Each selection method
for method, features in selections.items():
    auc, n = evaluate_features(list(features), method)
    results.append({"Method": method, "N Features": n, "Test AUC": auc})

# Strong consensus
auc_consensus, n_consensus = evaluate_features(strong_consensus, "Consensus (≥4 methods)")
results.append({"Method": "Consensus (≥4 methods)", "N Features": n_consensus, "Test AUC": auc_consensus})

df_perf = pd.DataFrame(results).round(4)
print("=" * 70)
print("PERFORMANCE COMPARISON: Feature Selection Methods")
print("=" * 70)
print(df_perf.to_string(index=False))

print("\n💡 Feature selection often IMPROVES AUC by removing noise")
print("   Even with 50% fewer features, performance is maintained or better")

In [ ]:
# ── 9.3 Visualization ────────────────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_perf["N Features"],
    y=df_perf["Test AUC"],
    mode="markers+text",
    text=df_perf["Method"],
    textposition="top center",
    marker=dict(
        size=15,
        color=df_perf["Test AUC"],
        colorscale="Viridis",
        line=dict(width=2, color="white"),
    ),
))

fig.update_layout(
    title="Feature Selection: Number of Features vs. Test AUC",
    xaxis_title="Number of Features",
    yaxis_title="Test ROC-AUC",
    height=450,
)
fig.show()

print("💡 The Pareto frontier: maximize AUC with minimum features")
print("   (top-left corner of the plot)")

---
## §10 · Key Business Takeaway

> ### 🎯 Action Items for a Data Scientist
>
> | Method | Type | Speed | Best For |
> |--------|------|-------|----------|
> | **Mutual Information** | Filter | Fast | Quick screening, non-linear relationships |
> | **RFE** | Wrapper | Slow | Small feature sets, model-specific |
> | **LASSO** | Embedded | Medium | Linear models, sparse solutions |
> | **XGBoost Importance** | Model-based | Fast | Tree-based models, quick insights |
> | **SHAP** | Game-theoretic | Medium | Interpretability, local explanations |
>
> ### 💡 Revolut-Specific Guidelines
>
> 1. **Start with mutual information** for fast, model-agnostic screening:
>    ```python
>    from sklearn.feature_selection import mutual_info_classif
>    mi_scores = mutual_info_classif(X, y)
>    ```
>
> 2. **Use SHAP for compliance and explainability**:
>    - Regulators require explanations for credit decisions.
>    - SHAP provides both global and local (per-prediction) explanations.
>
> 3. **LASSO for linear models** (e.g., scorecards):
>    - Automatically produces sparse models.
>    - Easy to deploy and interpret.
>
> 4. **Consensus approach** for production feature sets:
>    - Run 3–5 methods.
>    - Select features chosen by ≥3 methods.
>    - Reduces risk of missing important features.
>
> ### 📌 Feature Selection Workflow
>
> ```
> 1. Generate candidate features (500+)
> 2. Remove low-variance and highly correlated features
> 3. Apply mutual information (filter to top 100)
> 4. Apply LASSO or XGBoost importance (filter to top 50)
> 5. Use SHAP to validate and explain
> 6. Final validation: train model, measure AUC, compare to baseline
> ```
>
> ### 🔑 Key Takeaways
>
> 1. **More features ≠ better model** — noise degrades performance.
> 2. **Different methods select different features** — use consensus.
> 3. **SHAP is the gold standard** for interpretability (required for compliance).
> 4. **Feature selection reduces latency** — critical for real-time fraud scoring.
> 5. **Always validate** that selected features maintain model performance.